# ML Assignment 3 — Classification

## Objective
Evaluate understanding and ability to apply **supervised learning classification techniques** to the **Breast Cancer Wisconsin dataset** from `sklearn`.

---

## Table of Contents
1. [Loading and Preprocessing](#1-loading-and-preprocessing)
2. [Classification Algorithms](#2-classification-algorithms)
   - 2.1 Logistic Regression
   - 2.2 Decision Tree Classifier
   - 2.3 Random Forest Classifier
   - 2.4 Support Vector Machine (SVM)
   - 2.5 k-Nearest Neighbors (k-NN)
3. [Model Comparison](#3-model-comparison)
4. [Conclusion](#4-conclusion)

---
## 1. Loading and Preprocessing

### 1.1 About the Dataset

The **Breast Cancer Wisconsin (Diagnostic) dataset** is a classic binary classification dataset built into `sklearn`. It contains **569 samples** and **30 numeric features** computed from digitized images of fine needle aspirates (FNA) of breast masses. The features describe characteristics of cell nuclei such as:

- **Radius**, **Texture**, **Perimeter**, **Area**, **Smoothness**
- **Compactness**, **Concavity**, **Concave points**, **Symmetry**, **Fractal dimension**

Each feature has three measurements: **mean**, **standard error**, and **worst** (largest). The **target** is binary:
- `0` → **Malignant** (cancerous)
- `1` → **Benign** (non-cancerous)

### 1.2 Preprocessing Steps

| Step | Why It's Needed |
|------|----------------|
| **Check for missing values** | Ensure data integrity before modelling |
| **Train-test split** | Evaluate model on unseen data to prevent overfitting |
| **Feature Scaling (StandardScaler)** | Many algorithms (SVM, k-NN, Logistic Regression) are sensitive to feature magnitudes — scaling ensures equal contribution from all features |

In [ ]:
# ─── Import all required libraries ───────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

print("✅ All libraries imported successfully!")

In [ ]:
# ─── Load the dataset ────────────────────────────────────────────────────────
data = load_breast_cancer()

# Convert to DataFrame for easier exploration
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

print("📊 Dataset Shape:", df.shape)
print("\n📋 First 5 rows:")
df.head()

In [ ]:
# ─── Basic Dataset Information ───────────────────────────────────────────────
print("📌 Target Classes:", data.target_names)   # ['malignant', 'benign']
print("\n📌 Class Distribution:")
print(df['target'].value_counts().rename({0: 'Malignant', 1: 'Benign'}))

print("\n📌 Dataset Info:")
df.info()

In [ ]:
# ─── Check for Missing Values ────────────────────────────────────────────────
missing = df.isnull().sum()
print("🔍 Missing Values per Column:")
print(missing[missing > 0] if missing.any() else "✅ No missing values found! The dataset is clean.")

In [ ]:
# ─── Class Distribution Visualization ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = df['target'].value_counts()
axes[0].bar(['Malignant (0)', 'Benign (1)'], [counts[0], counts[1]],
            color=['#e74c3c', '#2ecc71'], edgecolor='black', width=0.5)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate([counts[0], counts[1]]):
    axes[0].text(i, v + 3, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie([counts[0], counts[1]], labels=['Malignant', 'Benign'],
            autopct='%1.1f%%', colors=['#e74c3c', '#2ecc71'],
            startangle=90, explode=(0.05, 0))
axes[1].set_title('Class Proportion', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ─── Feature Scaling & Train-Test Split ──────────────────────────────────────
X = data.data          # Features (30 columns)
y = data.target        # Target labels

# Split: 80% train, 20% test | random_state ensures reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# StandardScaler: (value - mean) / std  → mean=0, std=1 for each feature
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # Fit on train, transform train
X_test_scaled  = scaler.transform(X_test)         # Only transform test (no fit!)

print(f"✅ Training samples : {X_train_scaled.shape[0]}")
print(f"✅ Testing  samples : {X_test_scaled.shape[0]}")
print(f"✅ Number of features: {X_train_scaled.shape[1]}")
print(f"\n📌 Before Scaling — mean of feature 0: {X_train[:, 0].mean():.4f}")
print(f"📌 After  Scaling — mean of feature 0: {X_train_scaled[:, 0].mean():.6f}")

---
## 2. Classification Algorithms

We implement five supervised learning classifiers. A helper function is defined below to evaluate and store results for all models.

In [ ]:
# ─── Helper Function: Evaluate any classifier ────────────────────────────────
results = {}   # Dictionary to store all model results

def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    """
    Trains a model and prints evaluation metrics.
    Stores results in the global `results` dict.
    """
    model.fit(X_tr, y_tr)              # Train
    y_pred = model.predict(X_te)       # Predict on test set

    acc  = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred)
    rec  = recall_score(y_te, y_pred)
    f1   = f1_score(y_te, y_pred)
    cm   = confusion_matrix(y_te, y_pred)

    results[name] = {'Accuracy': acc, 'Precision': prec,
                     'Recall': rec, 'F1 Score': f1, 'CM': cm}

    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"\n  Classification Report:")
    print(classification_report(y_te, y_pred,
                                target_names=['Malignant','Benign']))

    # Confusion Matrix
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Malignant','Benign'],
                yticklabels=['Malignant','Benign'])
    plt.title(f'Confusion Matrix — {name}', fontweight='bold')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.show()

    return model

print("✅ Evaluation helper function ready.")

---
### 2.1 Logistic Regression

#### How It Works
Logistic Regression is a **linear classification** algorithm. Despite its name, it is used for **classification, not regression**. It models the **probability** that a sample belongs to a particular class using the **sigmoid (logistic) function**:

$$P(y=1 \mid x) = \frac{1}{1 + e^{-(\mathbf{w}^T\mathbf{x} + b)}}$$

- It learns weights `w` and bias `b` that best separate the two classes.
- If the predicted probability is **≥ 0.5 → Benign**, else **Malignant**.
- Optimized using **Maximum Likelihood Estimation** (minimizing log-loss).

#### Why Suitable for This Dataset?
- The dataset is **linearly separable to a large extent** — malignant and benign tumors differ significantly in features.
- Works well with **scaled features** (which we applied via StandardScaler).
- **Interpretable** — the weights tell us which features are most predictive.
- **Fast to train** even with 30 features.
- **Regularization** (`C` parameter) prevents overfitting.

In [ ]:
# ─── 2.1 Logistic Regression ─────────────────────────────────────────────────
# max_iter=10000  → increase iterations to ensure convergence on this dataset
# C=1.0           → inverse of regularization strength (default)
# random_state=42 → reproducibility

lr_model = LogisticRegression(max_iter=10000, C=1.0, random_state=42)

# Uses SCALED data — LR is sensitive to feature magnitudes
lr_model = evaluate_model(
    'Logistic Regression',
    lr_model, X_train_scaled, X_test_scaled, y_train, y_test
)

In [ ]:
# ─── Feature Coefficients (Interpretability of LR) ────────────────────────────
coef_df = pd.DataFrame({
    'Feature': data.feature_names,
    'Coefficient': lr_model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False).head(10)

plt.figure(figsize=(9, 4))
colors = ['#2ecc71' if c > 0 else '#e74c3c' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='black')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Top 10 Feature Coefficients — Logistic Regression', fontweight='bold')
plt.xlabel('Coefficient Value')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

---
### 2.2 Decision Tree Classifier

#### How It Works
A Decision Tree builds a **flowchart-like tree structure** by recursively splitting the dataset based on the feature that provides the **best split** (using **Gini impurity** or **Entropy/Information Gain**).

- **Root Node** → entire dataset
- **Internal Nodes** → feature-based questions (e.g., `mean radius > 14.5?`)
- **Leaf Nodes** → final class predictions

**Gini Impurity** formula (default criterion):
$$Gini = 1 - \sum_{i=1}^{C} p_i^2$$
where $p_i$ is the proportion of class $i$ at the node.

The tree grows until all leaves are pure or a stopping criterion is met.

#### Why Suitable for This Dataset?
- Handles **non-linear boundaries** between malignant and benign tumors.
- **No feature scaling required** — splits are based on thresholds, not distances.
- **Highly interpretable** — you can visualize the exact decision path.
- Prone to **overfitting** if not constrained (`max_depth` helps control this).

In [ ]:
# ─── 2.2 Decision Tree Classifier ────────────────────────────────────────────
# max_depth=5    → limit tree depth to prevent overfitting
# criterion='gini' → use Gini impurity for splitting (default)
# random_state=42  → reproducibility

dt_model = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42)

# Decision Trees do NOT require scaled data, but we use unscaled X here.
# Using scaled data is also fine — it doesn't affect the splits.
dt_model = evaluate_model(
    'Decision Tree',
    dt_model, X_train, X_test, y_train, y_test
)

In [ ]:
# ─── Top Feature Importances from Decision Tree ───────────────────────────────
imp_df = pd.DataFrame({
    'Feature': data.feature_names,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

plt.figure(figsize=(9, 4))
plt.barh(imp_df['Feature'], imp_df['Importance'],
         color='#3498db', edgecolor='black')
plt.title('Top 10 Feature Importances — Decision Tree', fontweight='bold')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

---
### 2.3 Random Forest Classifier

#### How It Works
Random Forest is an **ensemble learning** method that builds **multiple Decision Trees** and combines their predictions. It uses two key techniques:

1. **Bagging (Bootstrap Aggregating)**: Each tree is trained on a **random subset** of training samples (with replacement).
2. **Feature Randomness**: At each split, only a **random subset of features** is considered (typically √n_features).

The final prediction is made by **majority voting** (classification) across all trees:
$$\hat{y} = \text{mode}(h_1(x), h_2(x), \ldots, h_T(x))$$

#### Why Suitable for This Dataset?
- **Reduces overfitting** compared to a single Decision Tree — crucial for medical data.
- **Robust to noise and outliers** in the feature measurements.
- Handles **correlated features** well (many breast cancer features are correlated).
- Provides **feature importances** which aid medical interpretation.
- Generally achieves **high accuracy** with minimal hyperparameter tuning.

In [ ]:
# ─── 2.3 Random Forest Classifier ────────────────────────────────────────────
# n_estimators=100  → build 100 decision trees in the forest
# max_depth=None    → trees grow until leaves are pure (forest handles overfitting)
# random_state=42   → reproducibility
# n_jobs=-1         → use all available CPU cores for speed

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# Random Forests do NOT require feature scaling
rf_model = evaluate_model(
    'Random Forest',
    rf_model, X_train, X_test, y_train, y_test
)

In [ ]:
# ─── Feature Importances — Random Forest ──────────────────────────────────────
rf_imp = pd.DataFrame({
    'Feature': data.feature_names,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

plt.figure(figsize=(9, 4))
plt.barh(rf_imp['Feature'], rf_imp['Importance'],
         color='#27ae60', edgecolor='black')
plt.title('Top 10 Feature Importances — Random Forest', fontweight='bold')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

---
### 2.4 Support Vector Machine (SVM)

#### How It Works
SVM finds the **optimal hyperplane** that separates two classes with the **maximum margin**. The data points closest to the hyperplane are called **support vectors**.

$$\text{Maximize: } \frac{2}{\|\mathbf{w}\|}  \quad \text{subject to: } y_i(\mathbf{w}^T\mathbf{x}_i + b) \geq 1$$

For **non-linearly separable** data, SVM uses the **kernel trick** — it maps data to a higher-dimensional space where it becomes linearly separable:

| Kernel | When to Use |
|--------|-------------|
| `linear` | Data is linearly separable |
| `rbf` (default) | Non-linear boundaries (most common) |
| `poly` | Polynomial decision boundaries |

The `C` parameter controls the **trade-off between margin width and misclassifications**.

#### Why Suitable for This Dataset?
- **Effective in high-dimensional spaces** — 30 features are well within SVM's strength.
- The **RBF kernel** handles non-linear separation between malignant and benign classes.
- **Robust** to overfitting in high-dimensional spaces.
- **Scaling is critical** for SVM — distances between points determine the support vectors.

In [ ]:
# ─── 2.4 Support Vector Machine (SVM) ────────────────────────────────────────
# kernel='rbf'        → Radial Basis Function kernel for non-linear separation
# C=1.0               → Regularization parameter (higher C = less regularization)
# gamma='scale'       → kernel coefficient = 1/(n_features * X.var())
# probability=False   → not computing probabilities (faster)
# random_state=42     → reproducibility

svm_model = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)

# SVM REQUIRES scaled data — distances matter!
svm_model = evaluate_model(
    'SVM',
    svm_model, X_train_scaled, X_test_scaled, y_train, y_test
)

---
### 2.5 k-Nearest Neighbors (k-NN)

#### How It Works
k-NN is a **lazy learning** (instance-based) algorithm — it **memorizes the entire training set** and makes predictions at test time by:

1. Computing the **distance** between the test point and all training points (usually **Euclidean distance**):
$$d(x, x') = \sqrt{\sum_{i=1}^{n}(x_i - x'_i)^2}$$

2. Finding the **k nearest neighbors**.
3. Assigning the **majority class** among those k neighbors.

**Effect of k:**
- **Small k** (e.g., k=1) → complex boundary, prone to overfitting/noise.
- **Large k** → smooth boundary, may underfit.
- **k=5** is a commonly used default.

#### Why Suitable for This Dataset?
- **Simple and intuitive** — similar tumors (feature-wise) tend to be in the same class.
- **No training phase** — great for quick prototyping.
- **Feature scaling is essential** — without it, features with larger ranges dominate the distance calculation.
- Works well when **decision boundary is irregular**.

In [ ]:
# ─── 2.5 k-Nearest Neighbors (k-NN) ──────────────────────────────────────────
# n_neighbors=5       → use 5 nearest neighbors (common default)
# metric='euclidean'  → Euclidean distance measure
# weights='uniform'   → all k neighbors vote equally

knn_model = KNeighborsClassifier(n_neighbors=5, metric='euclidean', weights='uniform')

# k-NN REQUIRES scaled data — purely distance-based
knn_model = evaluate_model(
    'k-NN',
    knn_model, X_train_scaled, X_test_scaled, y_train, y_test
)

In [ ]:
# ─── Finding Optimal k for k-NN ───────────────────────────────────────────────
k_range = range(1, 21)
k_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    k_scores.append(accuracy_score(y_test, knn.predict(X_test_scaled)))

plt.figure(figsize=(9, 4))
plt.plot(k_range, k_scores, marker='o', color='#9b59b6', linewidth=2)
plt.axvline(x=k_scores.index(max(k_scores)) + 1, color='red',
            linestyle='--', label=f'Best k={k_scores.index(max(k_scores))+1}')
plt.title('k-NN: Accuracy vs Number of Neighbors (k)', fontweight='bold')
plt.xlabel('k (Number of Neighbors)')
plt.ylabel('Accuracy')
plt.xticks(k_range)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_k = k_scores.index(max(k_scores)) + 1
print(f"✅ Best k = {best_k} with accuracy = {max(k_scores):.4f}")

---
## 3. Model Comparison

Now we compare all five classifiers across four key metrics:

| Metric | What It Measures |
|--------|------------------|
| **Accuracy** | Overall correct predictions |
| **Precision** | Of all predicted Benign, how many are actually Benign? |
| **Recall** | Of all actual Benign, how many did we correctly identify? |
| **F1 Score** | Harmonic mean of Precision and Recall (balance metric) |

> **In medical diagnosis, Recall is especially important** — we want to minimize **False Negatives** (missing a malignant cancer).

In [ ]:
# ─── Comparison Table ────────────────────────────────────────────────────────
comparison_df = pd.DataFrame({
    name: {
        'Accuracy':  vals['Accuracy'],
        'Precision': vals['Precision'],
        'Recall':    vals['Recall'],
        'F1 Score':  vals['F1 Score']
    }
    for name, vals in results.items()
}).T.sort_values('Accuracy', ascending=False)

# Format to 4 decimal places
comparison_df = comparison_df.round(4)

print("\n📊 Model Performance Comparison:")
print(comparison_df.to_string())

In [ ]:
# ─── Grouped Bar Chart: All metrics side by side ──────────────────────────────
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
models  = list(comparison_df.index)

x = np.arange(len(models))
width = 0.18
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

fig, ax = plt.subplots(figsize=(13, 5))
for i, (metric, color) in enumerate(zip(metrics, colors)):
    bars = ax.bar(x + i * width, comparison_df[metric],
                  width, label=metric, color=color, edgecolor='white')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.003,
                f"{bar.get_height():.3f}",
                ha='center', va='bottom', fontsize=7.5, rotation=45)

ax.set_xlabel('Classifier', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Classification Model Comparison — All Metrics', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(models, fontsize=11)
ax.set_ylim(0.88, 1.02)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Heatmap of Metrics ───────────────────────────────────────────────────────
plt.figure(figsize=(8, 4))
sns.heatmap(comparison_df.astype(float), annot=True, fmt='.4f',
            cmap='YlGnBu', linewidths=0.5,
            annot_kws={'size': 11})
plt.title('Model Performance Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Radar / Spider Chart ────────────────────────────────────────────────────
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

labels   = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
N        = len(labels)
angles   = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles  += angles[:1]   # close the loop
palette  = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

for i, (model, row) in enumerate(comparison_df.iterrows()):
    values = row[labels].tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2,
            color=palette[i], label=model)
    ax.fill(angles, values, alpha=0.08, color=palette[i])

ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=11)
ax.set_ylim(0.88, 1.0)
ax.set_title('Radar Chart — Model Comparison', fontsize=13,
             fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Best and Worst Models ───────────────────────────────────────────────────
best_model  = comparison_df['Accuracy'].idxmax()
worst_model = comparison_df['Accuracy'].idxmin()

print("🏆 Best Performing Model :")
print(f"   → {best_model}")
print(f"   Accuracy  : {comparison_df.loc[best_model, 'Accuracy']:.4f}")
print(f"   F1 Score  : {comparison_df.loc[best_model, 'F1 Score']:.4f}")

print("\n⚠️  Worst Performing Model :")
print(f"   → {worst_model}")
print(f"   Accuracy  : {comparison_df.loc[worst_model, 'Accuracy']:.4f}")
print(f"   F1 Score  : {comparison_df.loc[worst_model, 'F1 Score']:.4f}")

---
## 4. Conclusion

### Model Performance Summary

| Rank | Model | Strengths | Weaknesses |
|------|-------|-----------|------------|
| 🥇 1 | **SVM / Random Forest** | High accuracy, robust | Slower on large data / Less interpretable |
| 🥈 2 | **Logistic Regression** | Fast, interpretable, strong baseline | Linear boundary assumption |
| 🥉 3 | **k-NN** | Simple, no training phase | Slow at prediction, sensitive to noise |
| 4 | **Decision Tree** | Interpretable, fast | Prone to overfitting |

### Key Takeaways

1. **Feature Scaling** was critical for SVM and k-NN — without it, these models would perform poorly.
2. **SVM with RBF kernel** and **Random Forest** are generally the strongest performers on this dataset due to their ability to capture non-linear patterns.
3. **Logistic Regression** serves as an excellent baseline and is highly interpretable — important in medical domains.
4. **Decision Tree** is the most interpretable but prone to overfitting without depth constraints.
5. **Recall** is the most critical metric here — a **False Negative** (predicting Benign when it's Malignant) is far more dangerous than a **False Positive**.

### Recommendation
For **clinical deployment**, **SVM or Random Forest** would be recommended for their high accuracy and recall. However, **Logistic Regression** should be kept as a reference due to its interpretability for medical professionals.